# 09 — YOLO framing-invariance retrain: end-to-end Kaggle version

This notebook fixes the unsafe parts of the original notebook and makes every external dependency explicit.
It trains, evaluates, checks clean-wheel false positives, compares framing stability against the shipped
model, exports ONNX, and creates one downloadable ZIP.

## What “zooming out is not a real whole-car photograph” means

The synthetic zoom-out step shrinks a close-up image and places it on a larger neutral canvas. That teaches
the detector that damage may occupy fewer pixels, but it does **not** create the real information present in
a whole-car UAE photograph: actual body geometry, camera distance and perspective, road/dealership
backgrounds, glare, shadows, compression, occlusion, multiple panels, and realistic damage-to-car scale.
Therefore synthetic zoom-out can reduce framing instability, but it cannot fully remove the domain gap.

For a complete production fix, add genuinely labelled UAE whole-car images to training. This notebook has
an optional input for that dataset and keeps its test split held out.

## Required Kaggle inputs

Use these dataset structures and, ideally, these Kaggle slugs:

```text
/kaggle/input/damage-unified/
  data.yaml
  train/images, train/labels
  val/images,   val/labels
  test/images,  test/labels       # optional; a test split is created from val if absent

/kaggle/input/framing-auxiliary/
  clean/                          # genuinely undamaged cars/wheels
  wholecar/                       # genuine wide whole-car photos for stability evaluation

/kaggle/input/shipped-weights/
  best.pt                         # currently deployed model

/kaggle/input/yolov8s-pretrained/ # recommended when Kaggle Internet is disabled
  yolov8s.pt

/kaggle/input/uae-whole-car-labelled/   # optional but required for the complete fix
  data.yaml
  train/images, train/labels
  val/images,   val/labels
  test/images,  test/labels
```

The unified and UAE-labelled datasets must use this exact class order:

```text
0 dent
1 scratch
2 crack
3 glass_shatter
4 lamp_broken
5 tire_flat
6 punctured
7 missing_part
```

**Do not use damaged images in `clean/`.** When a separate clean-wheel evaluation folder is not supplied,
this notebook deterministically holds out part of `clean/` and never copies those held-out files into
training.

## 1 · Configuration — edit only this cell

In [ ]:
from pathlib import Path

# ---------- Required inputs ----------
UNIFIED_DATA_YAML = Path('/kaggle/input/damage-unified/data.yaml')
CLEAN_CAR_DIR = Path('/kaggle/input/framing-auxiliary/clean')
WHOLE_CAR_EVAL_DIR = Path('/kaggle/input/framing-auxiliary/wholecar')
OLD_WEIGHTS_PATH = Path('/kaggle/input/shipped-weights/best.pt')

# Recommended: attach this file so training does not depend on Kaggle Internet.
PRETRAINED_WEIGHTS_PATH = Path('/kaggle/input/yolov8s-pretrained/yolov8s.pt')

# Optional complete-fix dataset. Leave as None for augmentation-only training.
UAE_LABELLED_DATA_YAML = Path('/kaggle/input/uae-whole-car-labelled/data.yaml')

# Optional independent clean-wheel evaluation folder. When None/missing, 20% of CLEAN_CAR_DIR
# is held out before negative images are copied into training.
CLEAN_WHEEL_EVAL_DIR = None

# ---------- Run behaviour ----------
STRICT_PRODUCTION = True          # True: missing required inputs stop before expensive training
RESET_WORKING_DIR = True          # always start from a clean /kaggle/working copy
SEED = 42
TEST_FRACTION = 0.15
MIN_TEST_IMAGES_PER_CLASS = 1
CLEAN_EVAL_FRACTION = 0.20
MIN_CLEAN_EVAL_IMAGES = 20
NEGATIVE_FRACTION = 0.12          # target fraction after zoom augmentation
MAX_NEGATIVES = 1500
MAX_STABILITY_IMAGES = 30

# ---------- Training ----------
EPOCHS = 60
PATIENCE = 15
IMAGE_SIZE = 640
BATCH_SIZE = -1                   # Ultralytics automatic batch sizing; use 8/4 if auto sizing fails
WORKERS = 2

# Paste the result from the product repository's scripts/stability_check.py after running it.
# The complete shipping gate is <= 15. Leave None during the Kaggle model-training run.
PRODUCT_SCORE_RANGE = None

UNIFIED = [
    'dent', 'scratch', 'crack', 'glass_shatter',
    'lamp_broken', 'tire_flat', 'punctured', 'missing_part',
]

## 2 · Environment and GPU preflight

In [ ]:
# Keep Kaggle's preinstalled CUDA-enabled PyTorch. Replacing torch inside a running kernel is a
# common source of import/CUDA errors. Install only the libraries this notebook directly needs.
%pip install -q "ultralytics==8.4.95" "onnx>=1.16" "onnxruntime>=1.18" "onnxslim" "pyyaml>=6.0"

In [ ]:
import ast
import hashlib
import json
import math
import os
import random
import shutil
import sys
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import torch
import yaml
import ultralytics
from ultralytics import YOLO

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Python:', sys.version.split()[0])
print('torch:', torch.__version__)
print('ultralytics:', ultralytics.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA capability:', torch.cuda.get_device_capability(0))
else:
    raise RuntimeError(
        'GPU is not enabled. In Kaggle open Notebook options/Settings, select a GPU accelerator, '
        'restart the session, and run from the first cell.'
    )

## 3 · Resolve and validate every input before training

In [ ]:
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}


def image_files(folder: Path):
    if not folder or not folder.exists():
        return []
    return sorted(
        p for p in folder.rglob('*')
        if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES
    )


def exact_names(config):
    names = config.get('names')
    if isinstance(names, dict):
        return [names[k] for k in sorted(names, key=lambda x: int(x))]
    return list(names or [])


def resolve_dataset_root(yaml_path: Path, config: dict):
    raw = config.get('path')
    if raw in (None, '', '.'):
        return yaml_path.parent
    root = Path(str(raw))
    if not root.is_absolute():
        root = yaml_path.parent / root
    return root


def split_image_dir(yaml_path: Path, config: dict, split: str):
    value = config.get(split)
    if not value:
        return None
    if isinstance(value, list):
        raise ValueError(f'{yaml_path}: list-valued {split} paths are not supported by this notebook')
    p = Path(str(value))
    if not p.is_absolute():
        p = resolve_dataset_root(yaml_path, config) / p
    return p


def paired_label_dir(image_dir: Path):
    if image_dir is None:
        return None
    parts = list(image_dir.parts)
    if 'images' not in parts:
        raise ValueError(
            f'Expected split image directory containing an "images" path component, got {image_dir}'
        )
    idx = len(parts) - 1 - parts[::-1].index('images')
    parts[idx] = 'labels'
    return Path(*parts)


def load_dataset_yaml(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f'{label} data.yaml not found: {path}')
    config = yaml.safe_load(path.read_text())
    names = exact_names(config)
    if names != UNIFIED:
        raise ValueError(
            f'{label} class order mismatch.\nFound:    {names}\nRequired: {UNIFIED}\n'
            'Correct the source mapping; never reorder UNIFIED in this notebook.'
        )
    return config


def validate_label_file(path: Path):
    errors = []
    if not path.exists():
        return ['missing label file']
    text = path.read_text().strip()
    if not text:
        return []
    for line_no, line in enumerate(text.splitlines(), 1):
        fields = line.split()
        if len(fields) != 5:
            errors.append(f'line {line_no}: expected 5 fields, got {len(fields)}')
            continue
        try:
            cls = int(fields[0])
            cx, cy, w, h = map(float, fields[1:])
        except ValueError:
            errors.append(f'line {line_no}: non-numeric YOLO label')
            continue
        if not 0 <= cls < len(UNIFIED):
            errors.append(f'line {line_no}: class id {cls} is outside 0..{len(UNIFIED)-1}')
        if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
            errors.append(f'line {line_no}: invalid normalized box {(cx, cy, w, h)}')
    return errors


def validate_split(yaml_path: Path, config: dict, split: str, require_nonempty: bool):
    img_dir = split_image_dir(yaml_path, config, split)
    if img_dir is None:
        if require_nonempty:
            raise ValueError(f'{yaml_path}: missing {split} entry')
        return {'images': 0, 'labelled_images': 0, 'instances': Counter(), 'image_dir': None}
    lbl_dir = paired_label_dir(img_dir)
    images = image_files(img_dir)
    if require_nonempty and not images:
        raise ValueError(f'{split} has no images: {img_dir}')

    instances = Counter()
    labelled_images = 0
    problems = []
    for img in images:
        rel = img.relative_to(img_dir)
        label = (lbl_dir / rel).with_suffix('.txt')
        errs = validate_label_file(label)
        if errs:
            problems.append(f'{img.name}: ' + '; '.join(errs[:3]))
            continue
        text = label.read_text().strip() if label.exists() else ''
        if text:
            labelled_images += 1
            for line in text.splitlines():
                instances[int(line.split()[0])] += 1

    if problems:
        preview = '\n'.join(problems[:20])
        raise ValueError(f'{split} label validation failed ({len(problems)} files):\n{preview}')

    return {
        'images': len(images),
        'labelled_images': labelled_images,
        'instances': instances,
        'image_dir': img_dir,
        'label_dir': lbl_dir,
    }


unified_cfg = load_dataset_yaml(UNIFIED_DATA_YAML, 'Unified')
unified_report = {
    split: validate_split(UNIFIED_DATA_YAML, unified_cfg, split, require_nonempty=(split != 'test'))
    for split in ('train', 'val', 'test')
}

clean_files_all = image_files(CLEAN_CAR_DIR)
wholecar_files = image_files(WHOLE_CAR_EVAL_DIR)

required_paths = {
    'clean-car negative folder': (CLEAN_CAR_DIR, bool(clean_files_all)),
    'whole-car stability folder': (WHOLE_CAR_EVAL_DIR, bool(wholecar_files)),
    'old shipped weights': (OLD_WEIGHTS_PATH, OLD_WEIGHTS_PATH.exists()),
}
for label, (path, ok) in required_paths.items():
    print(f'{label:<30} {"OK" if ok else "MISSING"}  {path}')

if STRICT_PRODUCTION:
    missing = [label for label, (_, ok) in required_paths.items() if not ok]
    if missing:
        raise FileNotFoundError(
            'Required production inputs are missing: ' + ', '.join(missing) + '. '
            'Attach the Kaggle datasets using the folder contract in the first cell.'
        )

print('\nUnified dataset:')
for split, report in unified_report.items():
    counts = {UNIFIED[k]: int(v) for k, v in sorted(report['instances'].items())}
    print(f'  {split:<5} images={report["images"]:<6} labelled={report["labelled_images"]:<6} instances={counts}')
print(f'Clean photos: {len(clean_files_all)}')
print(f'Whole-car stability photos: {len(wholecar_files)}')

uae_cfg = None
uae_available = UAE_LABELLED_DATA_YAML is not None and UAE_LABELLED_DATA_YAML.exists()
if uae_available:
    uae_cfg = load_dataset_yaml(UAE_LABELLED_DATA_YAML, 'UAE whole-car')
    uae_report = {
        split: validate_split(UAE_LABELLED_DATA_YAML, uae_cfg, split, require_nonempty=(split in ('train', 'val')))
        for split in ('train', 'val', 'test')
    }
    print('\nUAE labelled whole-car dataset found and validated:')
    for split, report in uae_report.items():
        print(f'  {split:<5} images={report["images"]:<6} instances={sum(report["instances"].values())}')
else:
    print('\nWARNING: no labelled UAE whole-car dataset is attached. This run is augmentation-only;')
    print('it can reduce the domain gap but cannot claim to be the complete whole-car fix.')

## 4 · Build a clean writable dataset and an honest test split

In [ ]:
WORK = Path('/kaggle/working/ds')
RUN_DIR = Path('/kaggle/working/runs/framing_invariance')
EXPORT_DIR = Path('/kaggle/working/export')

if RESET_WORKING_DIR:
    for path in (WORK, RUN_DIR, EXPORT_DIR):
        if path.exists():
            shutil.rmtree(path)

for split in ('train', 'val', 'test'):
    (WORK / split / 'images').mkdir(parents=True, exist_ok=True)
    (WORK / split / 'labels').mkdir(parents=True, exist_ok=True)


def safe_flat_name(relative_path: Path, prefix: str):
    token = '__'.join(relative_path.with_suffix('').parts)
    digest = hashlib.sha1(str(relative_path).encode()).hexdigest()[:8]
    return f'{prefix}{token}_{digest}'


def copy_split_into(yaml_path: Path, config: dict, split: str, destination_split: str, prefix: str):
    src_img = split_image_dir(yaml_path, config, split)
    if src_img is None or not src_img.exists():
        return 0
    src_lbl = paired_label_dir(src_img)
    copied = 0
    for img in image_files(src_img):
        rel = img.relative_to(src_img)
        label = (src_lbl / rel).with_suffix('.txt')
        stem = safe_flat_name(rel, prefix)
        out_img = WORK / destination_split / 'images' / f'{stem}{img.suffix.lower()}'
        out_lbl = WORK / destination_split / 'labels' / f'{stem}.txt'
        shutil.copy2(img, out_img)
        if label.exists():
            shutil.copy2(label, out_lbl)
        else:
            out_lbl.write_text('')
        copied += 1
    return copied


for split in ('train', 'val', 'test'):
    n = copy_split_into(UNIFIED_DATA_YAML, unified_cfg, split, split, prefix='base_')
    print(f'Copied unified {split}: {n}')


def classes_in_label(path: Path):
    text = path.read_text().strip() if path.exists() else ''
    return {int(line.split()[0]) for line in text.splitlines() if line.strip()}


def ensure_class_covered_test_split(frac=TEST_FRACTION, min_per_class=MIN_TEST_IMAGES_PER_CLASS):
    test_images = image_files(WORK / 'test' / 'images')
    if test_images:
        print(f'Existing test split retained: {len(test_images)} images')
        return

    val_dir = WORK / 'val' / 'images'
    val_lbl = WORK / 'val' / 'labels'
    val_images = image_files(val_dir)
    if not val_images:
        raise RuntimeError('Cannot create test split because validation split is empty')

    rng = random.Random(SEED)
    shuffled = val_images[:]
    rng.shuffle(shuffled)
    image_classes = {img: classes_in_label(val_lbl / f'{img.stem}.txt') for img in shuffled}
    available = Counter()
    for classes in image_classes.values():
        for cls in classes:
            available[cls] += 1

    missing_source_classes = [UNIFIED[i] for i in range(len(UNIFIED)) if available[i] == 0]
    if missing_source_classes:
        raise RuntimeError(
            'Cannot create an all-class held-out test split because validation has no examples of: '
            + ', '.join(missing_source_classes)
        )

    selected = []
    selected_set = set()
    class_image_counts = Counter()

    # Deterministic greedy coverage first.
    for cls in range(len(UNIFIED)):
        candidates = [img for img in shuffled if cls in image_classes[img] and img not in selected_set]
        while class_image_counts[cls] < min_per_class and candidates:
            best = max(
                candidates,
                key=lambda img: sum(class_image_counts[c] < min_per_class for c in image_classes[img]),
            )
            selected.append(best)
            selected_set.add(best)
            for c in image_classes[best]:
                class_image_counts[c] += 1
            candidates = [img for img in candidates if img not in selected_set]

    target = max(len(selected), int(math.ceil(len(val_images) * frac)))
    for img in shuffled:
        if len(selected) >= target:
            break
        if img not in selected_set:
            selected.append(img)
            selected_set.add(img)
            for c in image_classes[img]:
                class_image_counts[c] += 1

    for img in selected:
        label = val_lbl / f'{img.stem}.txt'
        shutil.move(str(img), WORK / 'test' / 'images' / img.name)
        shutil.move(str(label), WORK / 'test' / 'labels' / label.name)

    print(f'Created deterministic test split: {len(selected)} images (seed={SEED})')
    print('Test images per class:', {UNIFIED[i]: class_image_counts[i] for i in range(len(UNIFIED))})


ensure_class_covered_test_split()

### Optional: merge genuine labelled UAE whole-car train/validation images

In [ ]:
uae_train_added = 0
uae_val_added = 0
if uae_available:
    uae_train_added = copy_split_into(UAE_LABELLED_DATA_YAML, uae_cfg, 'train', 'train', prefix='uae_train_')
    uae_val_added = copy_split_into(UAE_LABELLED_DATA_YAML, uae_cfg, 'val', 'val', prefix='uae_val_')
    print(f'Added UAE whole-car images: train={uae_train_added}, val={uae_val_added}')
else:
    print('No UAE labelled whole-car images merged.')


def split_counts(split):
    imgs = image_files(WORK / split / 'images')
    counts = Counter()
    image_counts = Counter()
    for img in imgs:
        classes = classes_in_label(WORK / split / 'labels' / f'{img.stem}.txt')
        for cls in classes:
            image_counts[cls] += 1
        text = (WORK / split / 'labels' / f'{img.stem}.txt').read_text().strip()
        for line in text.splitlines() if text else []:
            counts[int(line.split()[0])] += 1
    return len(imgs), image_counts, counts

for split in ('train', 'val', 'test'):
    n, image_counts, instance_counts = split_counts(split)
    print(f'\n{split}: {n} images')
    print('  images/class:', {UNIFIED[i]: image_counts[i] for i in range(len(UNIFIED))})
    print('  boxes/class :', {UNIFIED[i]: instance_counts[i] for i in range(len(UNIFIED))})

# A test split missing a class cannot support an honest 8-class report.
_, test_image_counts, _ = split_counts('test')
missing_test = [UNIFIED[i] for i in range(len(UNIFIED)) if test_image_counts[i] == 0]
if missing_test:
    raise RuntimeError('Held-out test split is missing classes: ' + ', '.join(missing_test))

## 5 · Zoom-out augmentation

This produces **scale augmentation**, not a realistic whole-car scene. It is applied only to the original
close-up training images. Genuine UAE whole-car files are not synthetically zoomed out again.

In [ ]:
ZOOM_FACTORS = (0.55, 0.35)
MIN_BOX_AREA = 1e-4


def read_boxes(path: Path):
    text = path.read_text().strip() if path.exists() else ''
    boxes = []
    for line in text.splitlines() if text else []:
        cls, cx, cy, w, h = line.split()
        boxes.append((int(cls), float(cx), float(cy), float(w), float(h)))
    return boxes


def write_boxes(path: Path, boxes):
    path.write_text(''.join(
        f'{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n'
        for cls, cx, cy, w, h in boxes
    ))


def zoom_out(image, boxes, factor, rng):
    h, w = image.shape[:2]
    new_w = max(1, int(round(w * factor)))
    new_h = max(1, int(round(h * factor)))
    scale_x = new_w / w
    scale_y = new_h / h
    small = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.full_like(image, 114)
    ox_px = rng.randint(0, w - new_w)
    oy_px = rng.randint(0, h - new_h)
    canvas[oy_px:oy_px + new_h, ox_px:ox_px + new_w] = small
    ox = ox_px / w
    oy = oy_px / h

    output = []
    for cls, cx, cy, bw, bh in boxes:
        ncx = ox + cx * scale_x
        ncy = oy + cy * scale_y
        nbw = bw * scale_x
        nbh = bh * scale_y
        if nbw * nbh < MIN_BOX_AREA:
            continue
        if not (0 <= ncx <= 1 and 0 <= ncy <= 1 and 0 < nbw <= 1 and 0 < nbh <= 1):
            continue
        output.append((cls, ncx, ncy, nbw, nbh))
    return canvas, output


rng = random.Random(SEED)
train_img_dir = WORK / 'train' / 'images'
train_lbl_dir = WORK / 'train' / 'labels'
originals = [
    p for p in image_files(train_img_dir)
    if '_zoom' not in p.stem and not p.stem.startswith('uae_')
]
added = 0
skipped = 0
for image_path in originals:
    boxes = read_boxes(train_lbl_dir / f'{image_path.stem}.txt')
    if not boxes:
        continue
    image = cv2.imread(str(image_path))
    if image is None:
        skipped += 1
        continue
    for factor in ZOOM_FACTORS:
        zoomed, zoomed_boxes = zoom_out(image, boxes, factor, rng)
        if not zoomed_boxes:
            skipped += 1
            continue
        stem = f'{image_path.stem}_zoom{int(factor * 100)}'
        out_image = train_img_dir / f'{stem}.jpg'
        out_label = train_lbl_dir / f'{stem}.txt'
        ok = cv2.imwrite(str(out_image), zoomed, [cv2.IMWRITE_JPEG_QUALITY, 92])
        if not ok:
            raise IOError(f'Failed to write augmented image: {out_image}')
        write_boxes(out_label, zoomed_boxes)
        added += 1

print(f'Zoom-out images added: {added}; skipped: {skipped}')
if added == 0:
    raise RuntimeError('No zoom-out images were generated. Check training labels and image readability.')
print('Training images after zoom augmentation:', len(image_files(train_img_dir)))

## 6 · Clean-car negatives and a held-out clean-wheel gate

The detector needs examples of normal wheels and undamaged body panels. Those images are copied into the
training split with **empty YOLO label files**, explicitly marking them as background. A separate clean set
is held out and used later to measure the maximum `tire_flat` confidence. This prevents gate 2 from being
measured on images the model already saw during training.

In [ ]:
all_clean = image_files(CLEAN_CAR_DIR)
if not all_clean:
    raise RuntimeError(f'No clean-car images found under {CLEAN_CAR_DIR}')

rng = random.Random(SEED)
shuffled_clean = all_clean[:]
rng.shuffle(shuffled_clean)

configured_eval_dir = Path(CLEAN_WHEEL_EVAL_DIR) if CLEAN_WHEEL_EVAL_DIR else None
if configured_eval_dir and configured_eval_dir.exists():
    clean_eval_files = image_files(configured_eval_dir)
    clean_train_candidates = shuffled_clean
else:
    eval_count = max(MIN_CLEAN_EVAL_IMAGES, int(math.ceil(len(shuffled_clean) * CLEAN_EVAL_FRACTION)))
    if eval_count >= len(shuffled_clean):
        raise RuntimeError(
            f'Need more clean photos. Found {len(shuffled_clean)}, but at least one training negative and '
            f'{MIN_CLEAN_EVAL_IMAGES} held-out evaluation images are required.'
        )
    clean_eval_files = shuffled_clean[:eval_count]
    clean_train_candidates = shuffled_clean[eval_count:]

if len(clean_eval_files) < MIN_CLEAN_EVAL_IMAGES:
    raise RuntimeError(
        f'Held-out clean-wheel set has {len(clean_eval_files)} images; need at least {MIN_CLEAN_EVAL_IMAGES}.'
    )

positive_count = sum(
    1 for img in image_files(train_img_dir)
    if (train_lbl_dir / f'{img.stem}.txt').read_text().strip()
)
target_negatives = min(
    MAX_NEGATIVES,
    max(1, int(round((NEGATIVE_FRACTION / (1 - NEGATIVE_FRACTION)) * positive_count))),
)

if len(clean_train_candidates) < target_negatives:
    message = (
        f'Only {len(clean_train_candidates)} clean training images are available, but the configured '
        f'{NEGATIVE_FRACTION:.0%} target needs {target_negatives}. '
    )
    if STRICT_PRODUCTION:
        raise RuntimeError(message + 'Attach more clean-car photos or lower NEGATIVE_FRACTION deliberately.')
    print('WARNING:', message + 'Using all available clean training images.')
    target_negatives = len(clean_train_candidates)

negative_sources = clean_train_candidates[:target_negatives]
for index, source in enumerate(negative_sources):
    digest = hashlib.sha1(str(source).encode()).hexdigest()[:10]
    out_image = train_img_dir / f'negative_{index:05d}_{digest}.jpg'
    image = cv2.imread(str(source))
    if image is None:
        raise IOError(f'Unreadable clean image: {source}')
    ok = cv2.imwrite(str(out_image), image, [cv2.IMWRITE_JPEG_QUALITY, 92])
    if not ok:
        raise IOError(f'Failed to write negative image: {out_image}')
    (train_lbl_dir / f'{out_image.stem}.txt').write_text('')

clean_eval_manifest = Path('/kaggle/working/clean_eval_manifest.txt')
clean_eval_manifest.write_text('\n'.join(str(p) for p in clean_eval_files) + '\n')

print(f'Clean negatives added to training: {len(negative_sources)}')
print(f'Clean images held out for gate 2: {len(clean_eval_files)}')
print('Manifest:', clean_eval_manifest)

## 7 · Write final training YAML and run final preflight

In [ ]:
DATA_YAML = WORK / 'data.yaml'
DATA_YAML.write_text(yaml.safe_dump({
    'path': str(WORK),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'names': {i: name for i, name in enumerate(UNIFIED)},
}, sort_keys=False))

# Remove stale Ultralytics caches after creating/changing labels.
for cache in WORK.rglob('*.cache'):
    cache.unlink()

print(DATA_YAML.read_text())
for split in ('train', 'val', 'test'):
    images = image_files(WORK / split / 'images')
    labels = list((WORK / split / 'labels').glob('*.txt'))
    print(f'{split:<5}: {len(images)} images, {len(labels)} label files')
    if len(images) != len(labels):
        raise RuntimeError(f'{split}: image/label count mismatch')

## 8 · Train

In [ ]:
if PRETRAINED_WEIGHTS_PATH.exists():
    pretrained = str(PRETRAINED_WEIGHTS_PATH)
    print('Using attached pretrained weights:', pretrained)
else:
    pretrained = 'yolov8s.pt'
    print('Attached yolov8s.pt not found. Ultralytics will try to download it.')
    print('If Kaggle Internet is disabled, attach /kaggle/input/yolov8s-pretrained/yolov8s.pt.')

model = YOLO(pretrained)
try:
    results = model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        patience=PATIENCE,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=0,
        workers=WORKERS,
        seed=SEED,
        deterministic=True,
        scale=0.9,
        mosaic=1.0,
        close_mosaic=10,
        copy_paste=0.3,
        translate=0.2,
        fliplr=0.5,
        flipud=0.0,
        degrees=5.0,
        project='/kaggle/working/runs',
        name='framing_invariance',
        exist_ok=True,
        plots=True,
        verbose=True,
    )
except RuntimeError as exc:
    if 'out of memory' in str(exc).lower():
        raise RuntimeError(
            'CUDA out of memory. Change BATCH_SIZE in the configuration cell to 8, then 4 if needed, '
            'restart the session, and run all cells again.'
        ) from exc
    raise

BEST = Path(model.trainer.best)
if not BEST.exists():
    fallback = RUN_DIR / 'weights' / 'best.pt'
    if fallback.exists():
        BEST = fallback
if not BEST.exists():
    raise FileNotFoundError('Training completed but best.pt was not found')
print('Best weights:', BEST)

## 9 · Honest held-out evaluation with correct class-index mapping

In [ ]:
best = YOLO(str(BEST))
metrics = best.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=IMAGE_SIZE,
    device=0,
    plots=True,
    verbose=True,
)

metric_class_ids = [int(x) for x in metrics.box.ap_class_index]
position_for_class = {class_id: pos for pos, class_id in enumerate(metric_class_ids)}

per_class = {}
print(f'\nHELD-OUT TEST mAP@0.5={float(metrics.box.map50):.4f} '
      f'mAP@0.5:0.95={float(metrics.box.map):.4f}')
print(f'{"class":<16}{"P":>10}{"R":>10}{"mAP50":>10}')
for class_id, name in enumerate(UNIFIED):
    pos = position_for_class.get(class_id)
    if pos is None:
        print(f'{name:<16}{"-":>10}{"-":>10}{"-":>10}  absent from test split')
        continue
    precision = float(metrics.box.p[pos])
    recall = float(metrics.box.r[pos])
    map50 = float(metrics.box.ap50[pos])
    per_class[name] = {'precision': precision, 'recall': recall, 'mAP50': map50}
    print(f'{name:<16}{precision:>10.3f}{recall:>10.3f}{map50:>10.3f}')

classes_reported = len(per_class)
print(f'Classes with published P/R: {classes_reported}/8')

uae_test_metrics = None
if uae_available and uae_cfg.get('test'):
    uae_test_metrics = best.val(
        data=str(UAE_LABELLED_DATA_YAML),
        split='test',
        imgsz=IMAGE_SIZE,
        device=0,
        plots=False,
        verbose=False,
    )
    print(f'UAE-ONLY held-out test mAP@0.5={float(uae_test_metrics.box.map50):.4f} '
          f'mAP@0.5:0.95={float(uae_test_metrics.box.map):.4f}')
else:
    print('UAE-only held-out metrics: NOT RUN (no labelled UAE test split attached)')

## 10 · Gate 2: held-out clean-wheel `tire_flat` false positives

In [ ]:
TIRE_FLAT_CLASS_ID = UNIFIED.index('tire_flat')
clean_wheel_rows = []
for path in clean_eval_files:
    result = best.predict(
        source=str(path),
        imgsz=IMAGE_SIZE,
        conf=0.001,
        classes=[TIRE_FLAT_CLASS_ID],
        device=0,
        verbose=False,
    )[0]
    confidences = result.boxes.conf.detach().cpu().tolist() if len(result.boxes) else []
    max_conf = max(confidences, default=0.0)
    clean_wheel_rows.append({'image': str(path), 'max_tire_flat_confidence': float(max_conf)})

clean_wheel_rows.sort(key=lambda row: row['max_tire_flat_confidence'], reverse=True)
max_tire_flat_conf = clean_wheel_rows[0]['max_tire_flat_confidence'] if clean_wheel_rows else 1.0
num_clean_failures = sum(row['max_tire_flat_confidence'] >= 0.40 for row in clean_wheel_rows)

print(f'Max tire_flat confidence on held-out clean images: {max_tire_flat_conf:.4f}')
print(f'Images at or above 0.40: {num_clean_failures}/{len(clean_wheel_rows)}')
print('Worst examples:')
for row in clean_wheel_rows[:10]:
    print(f'  {row["max_tire_flat_confidence"]:.4f}  {row["image"]}')

## 11 · Framing stability on genuine whole-car photographs

The stability set is deliberately separate from the synthetic zoom-out training images. Labels are not
required for the consistency comparison, although a labelled UAE dataset is required to measure real
whole-car accuracy and to train the complete fix.

In [ ]:
CONF_GATE = 0.20


def framing_variants(image):
    h, w = image.shape[:2]
    yield 'original', image
    for percent in (3, 6, 10):
        dx = int(w * percent / 100)
        dy = int(h * percent / 100)
        cropped = image[dy:h-dy, dx:w-dx]
        if cropped.size:
            yield f'crop_{percent}', cropped
    for percent in (10, 25):
        px = int(w * percent / 100)
        py = int(h * percent / 100)
        yield f'back_{percent}', cv2.copyMakeBorder(
            image, py, py, px, px, cv2.BORDER_REPLICATE
        )
    yield 'resize_70', cv2.resize(image, (max(1, int(w * 0.7)), max(1, int(h * 0.7))))


def jaccard(a, b):
    union = a | b
    return len(a & b) / len(union) if union else 1.0


def stability_report(weights: Path, paths):
    detector = YOLO(str(weights))
    rows = []
    for path in paths:
        image = cv2.imread(str(path))
        if image is None:
            continue
        variants = []
        for name, variant in framing_variants(image):
            result = detector.predict(
                variant, imgsz=IMAGE_SIZE, conf=CONF_GATE, device=0, verbose=False
            )[0]
            classes = {UNIFIED[int(x)] for x in result.boxes.cls.detach().cpu().tolist()} \
                if len(result.boxes) else set()
            variants.append({
                'name': name,
                'classes': sorted(classes),
                'box_count': int(len(result.boxes)),
            })
        sets = [set(v['classes']) for v in variants]
        original = sets[0]
        mean_vs_original = float(np.mean([jaccard(original, s) for s in sets[1:]])) if len(sets) > 1 else 1.0
        union = set().union(*sets) if sets else set()
        intersection = set.intersection(*sets) if sets else set()
        all_variant_jaccard = len(intersection) / len(union) if union else 1.0
        rows.append({
            'image': str(path),
            'mean_jaccard_vs_original': mean_vs_original,
            'all_variant_jaccard': all_variant_jaccard,
            'variants': variants,
        })
    return rows


stability_paths = wholecar_files[:MAX_STABILITY_IMAGES]
if not stability_paths:
    raise RuntimeError('No genuine whole-car stability images are available')

old_model_for_order = YOLO(str(OLD_WEIGHTS_PATH))
old_names = old_model_for_order.names
old_order = [old_names[k] for k in sorted(old_names)] if isinstance(old_names, dict) else list(old_names)
if old_order != UNIFIED:
    raise RuntimeError(
        f'Old shipped model class order mismatch. Found {old_order}; required {UNIFIED}. '
        'The old/new stability comparison would be invalid.'
    )
del old_model_for_order

old_stability_rows = stability_report(OLD_WEIGHTS_PATH, stability_paths)
new_stability_rows = stability_report(BEST, stability_paths)

old_mean_agreement = float(np.mean([r['mean_jaccard_vs_original'] for r in old_stability_rows]))
new_mean_agreement = float(np.mean([r['mean_jaccard_vs_original'] for r in new_stability_rows]))
old_allway_agreement = float(np.mean([r['all_variant_jaccard'] for r in old_stability_rows]))
new_allway_agreement = float(np.mean([r['all_variant_jaccard'] for r in new_stability_rows]))

print(f'OLD mean Jaccard vs original: {old_mean_agreement:.4f}')
print(f'NEW mean Jaccard vs original: {new_mean_agreement:.4f}')
print(f'OLD all-variant Jaccard:      {old_allway_agreement:.4f}')
print(f'NEW all-variant Jaccard:      {new_allway_agreement:.4f}')
print(f'Improvement vs original:      {new_mean_agreement - old_mean_agreement:+.4f}')

## 12 · Export and verify ONNX

These settings preserve the existing application contract: fixed `[1,3,640,640]` input, no built-in NMS,
8 classes, and raw output `[1,12,8400]`.

In [ ]:
onnx_path = Path(YOLO(str(BEST)).export(
    format='onnx',
    imgsz=IMAGE_SIZE,
    opset=12,
    dynamic=False,
    nms=False,
    simplify=True,
    batch=1,
))
print('Exported:', onnx_path)

import onnxruntime as ort

session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
model_meta = session.get_modelmeta().custom_metadata_map
onnx_input = session.get_inputs()[0]
onnx_output = session.get_outputs()[0]

raw_names = model_meta.get('names')
if not raw_names:
    raise RuntimeError('ONNX metadata does not contain class names')
parsed_names = ast.literal_eval(raw_names)
if isinstance(parsed_names, dict):
    exported_order = [parsed_names.get(i, parsed_names.get(str(i))) for i in range(len(parsed_names))]
else:
    exported_order = list(parsed_names)

print('Input :', onnx_input.name, onnx_input.shape)
print('Output:', onnx_output.name, onnx_output.shape)
print('Class order:', exported_order)

assert list(onnx_input.shape) == [1, 3, IMAGE_SIZE, IMAGE_SIZE]
assert list(onnx_output.shape) == [1, 12, 8400]
assert exported_order == UNIFIED

# Runtime smoke test catches malformed exports that can be parsed but not executed.
dummy = np.zeros((1, 3, IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
smoke_output = session.run([onnx_output.name], {onnx_input.name: dummy})[0]
assert list(smoke_output.shape) == [1, 12, 8400]

onnx_sha256 = hashlib.sha256(onnx_path.read_bytes()).hexdigest()
print('ONNX runtime smoke test passed')
print('SHA-256:', onnx_sha256)

## 13 · Save reports and create one downloadable ZIP

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(BEST, EXPORT_DIR / 'best.pt')
shutil.copy2(onnx_path, EXPORT_DIR / 'best.onnx')
shutil.copy2(DATA_YAML, EXPORT_DIR / 'data.yaml')
shutil.copy2(clean_eval_manifest, EXPORT_DIR / 'clean_eval_manifest.txt')

for name in ('results.csv', 'results.png', 'confusion_matrix.png', 'PR_curve.png', 'P_curve.png', 'R_curve.png', 'F1_curve.png', 'args.yaml'):
    source = RUN_DIR / name
    if source.exists():
        shutil.copy2(source, EXPORT_DIR / name)

report = {
    'seed': SEED,
    'unified_class_order': UNIFIED,
    'used_labelled_uae_whole_car_training': bool(uae_train_added),
    'uae_train_images_added': int(uae_train_added),
    'uae_val_images_added': int(uae_val_added),
    'held_out_test': {
        'map50': float(metrics.box.map50),
        'map50_95': float(metrics.box.map),
        'classes_reported': int(classes_reported),
        'per_class': per_class,
    },
    'uae_only_test': None if uae_test_metrics is None else {
        'map50': float(uae_test_metrics.box.map50),
        'map50_95': float(uae_test_metrics.box.map),
    },
    'clean_wheel_gate': {
        'images': len(clean_wheel_rows),
        'maximum_tire_flat_confidence': float(max_tire_flat_conf),
        'images_at_or_above_0_40': int(num_clean_failures),
        'rows': clean_wheel_rows,
    },
    'framing_stability': {
        'images': len(stability_paths),
        'old_mean_jaccard_vs_original': old_mean_agreement,
        'new_mean_jaccard_vs_original': new_mean_agreement,
        'old_all_variant_jaccard': old_allway_agreement,
        'new_all_variant_jaccard': new_allway_agreement,
        'old_rows': old_stability_rows,
        'new_rows': new_stability_rows,
    },
    'product_score_range': PRODUCT_SCORE_RANGE,
    'onnx': {
        'input_shape': list(onnx_input.shape),
        'output_shape': list(onnx_output.shape),
        'class_order': exported_order,
        'sha256': onnx_sha256,
    },
}

(EXPORT_DIR / 'evaluation_report.json').write_text(json.dumps(report, indent=2))

zip_path = Path('/kaggle/working/framing_invariance_export.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(EXPORT_DIR.rglob('*')):
        if file.is_file():
            archive.write(file, file.relative_to(EXPORT_DIR))

print('Export directory:', EXPORT_DIR)
for file in sorted(EXPORT_DIR.iterdir()):
    print(f'  {file.name:<30} {file.stat().st_size:>12,} bytes')
print('Download ZIP:', zip_path)

## 14 · Exit gates — failing or not-run gates block production shipment

In [ ]:
def gate_status(value):
    if value is None:
        return 'NOT RUN'
    return 'PASS' if bool(value) else 'FAIL'

score_gate = None if PRODUCT_SCORE_RANGE is None else PRODUCT_SCORE_RANGE <= 15
uae_data_gate = bool(uae_train_added) and uae_test_metrics is not None

GATES = [
    ('mAP@0.5 on held-out test >= 0.60', float(metrics.box.map50) >= 0.60),
    ('all 8 classes have published P/R', classes_reported == 8),
    ('crack recall >= 0.45', per_class.get('crack', {}).get('recall', 0.0) >= 0.45),
    ('held-out clean-wheel max tire_flat confidence < 0.40', max_tire_flat_conf < 0.40),
    ('new framing agreement is not worse than shipped model', new_mean_agreement >= old_mean_agreement),
    ('genuine labelled UAE whole-car train and held-out test used', uae_data_gate),
    ('product score range across framing variants <= 15', score_gate),
    ('ONNX input is [1,3,640,640]', list(onnx_input.shape) == [1, 3, 640, 640]),
    ('ONNX output is [1,12,8400]', list(onnx_output.shape) == [1, 12, 8400]),
    ('ONNX class order unchanged', exported_order == UNIFIED),
]

width = max(len(name) for name, _ in GATES)
for name, value in GATES:
    print(f'{gate_status(value):<8} {name:<{width}}')

all_passed = all(value is True for _, value in GATES)
print('\nFINAL STATUS:', 'READY TO SHIP' if all_passed else 'NOT READY TO SHIP')
if PRODUCT_SCORE_RANGE is None:
    print('Run the repository command: python scripts/stability_check.py <whole-car-photo-directory>')
    print('Then set PRODUCT_SCORE_RANGE, run the configuration cell, and rerun this gate cell.')
if not uae_data_gate:
    print('The augmentation-only model may be useful, but the complete domain-gap fix remains blocked')
    print('until genuine labelled UAE whole-car train/validation/test data is attached and used.')

## 15 · Kaggle run order and download

1. Attach the four required inputs and the optional labelled UAE dataset.
2. Enable a GPU accelerator. Internet may remain off when `yolov8s.pt` is attached.
3. Edit only the configuration cell if your Kaggle dataset slugs differ.
4. Use **Restart session**, then **Run all** from the first cell.
5. Do not rerun only the data-splitting cells after a partial run; this notebook deletes and rebuilds the
   writable dataset when `RESET_WORKING_DIR=True`.
6. When all executable cells finish, download:
   `/kaggle/working/framing_invariance_export.zip`.
7. Use Kaggle **Save Version / Save & Run All** for the final reproducible run.
8. In the product repository, run the existing score-level stability script. Paste its score range into
   `PRODUCT_SCORE_RANGE` and rerun the exit-gates cell. A value above 15 blocks shipment.